# Metasyn Tutorial: Time-Series Dependencies and Logical Constraints

In this tutorial, we explore how to model time-series data with start and end dates, and how to add column dependency relationships and constraints to synthetic data using the implemented dunder methods.


### 0. Install and import

First, let's install metasyn if you haven't done so already.

In [1]:
# %pip install metasyn

Now let's import the packages we need.

In [2]:
import polars as pl

from metasyn import demo_data
from metasyn.builder import MetaFrameBuilder
from metasyn.distribution import DiscreteTruncatedNormalDistribution
from metasyn.distribution.base import (
    ColumnReference,
    IfThenElse,
)

### 1. Synthesising time-series data

A common challenge with time-series datasets is that columns can be related to each other. For example, in a hospital admissions dataset the `discharge_date` should always come *after* the `admission_date`. By default, metasyn treats each column independently, so this constraint is not preserved.

Let's load our hospital admissions dataset to see this in action.

In [3]:
df = demo_data("hospital_admissions")

df.head()


Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,str
1,2023-01-04 00:00:00,2023-01-06 00:00:00,2,44,164,67,"""F"""
2,2023-01-08 00:00:00,2023-01-18 00:00:00,10,78,154,57,"""F"""
3,2023-01-08 00:00:00,2023-01-19 00:00:00,11,18,167,77,"""M"""
4,2023-01-08 00:00:00,2023-01-24 00:00:00,16,83,177,95,"""F"""
5,2023-01-15 00:00:00,2023-01-21 00:00:00,6,9,138,25,"""M"""


If we fit and synthesise without any additional instructions, the dates are generated independently. Let's check how often this produces inconsistent results.

In [4]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

synth = builder.fit().synthesize()

n_inconsistent = (synth["Discharge_date"] < synth["Admission_date"]).sum()
print(f"\nRows where Discharge_date < Admission_date: {n_inconsistent} / {len(synth)}")

synth.head()

  Patient_id: 100%|██████████| 8/8 [00:00<00:00, 469.79variables/s]


Rows where Discharge_date < Admission_date: 55 / 100


Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,str
1,2023-05-24 00:00:00,2023-04-25 00:00:00,10,70,189,79,"""F"""
2,2024-03-06 00:00:00,2024-02-08 00:00:00,11,43,195,60,"""F"""
3,2024-01-18 00:00:00,2023-07-07 00:00:00,10,27,154,91,"""F"""
4,2024-08-28 00:00:00,2023-07-18 00:00:00,4,84,158,104,"""F"""
5,2024-11-08 00:00:00,2024-08-23 00:00:00,16,25,188,88,"""M"""


As we can see, there are some inconsistent rows. To fix this, we can add a hidden `duration_days` column that holds the difference in days between `Discharge_date` and `Admission_date`. Metasyn fits a distribution to the new column, which can then be used to compute `Discharge_date = Admission_date + duration_days`. Setting `hidden=True` ensures the helper column does **not** appear in the final output.

Note that `ColumnReference` refers to the values in a column at synthesis time, and can be used to build expressions between columns.

In [5]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

# Add a column and mark as hidden
builder.add_column("duration_days", hidden=True)

# Create a series for the difference between discharge_date and admission_date, and use it to derive discharge_date
builder["duration_days"].series = ColumnReference("Discharge_date") - ColumnReference("Admission_date")
builder["Discharge_date"].distribution = ColumnReference("Admission_date") + ColumnReference("duration_days")

synth = builder.fit().synthesize()

n_inconsistent = (synth["Discharge_date"] < synth["Admission_date"]).sum()
print(f"\nRows where Discharge_date < Admission_date: {n_inconsistent} / {len(synth)}")

synth

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 521.56variables/s]


Rows where Discharge_date < Admission_date: 0 / 100


Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,str
1,2024-06-09 00:00:00,2024-06-29 00:00:00,16,30,185,12,"""M"""
2,2024-01-31 00:00:00,2024-02-09 00:00:00,14,20,124,74,"""M"""
3,2023-02-15 00:00:00,2023-02-25 00:00:00,11,6,157,119,"""F"""
4,2023-03-05 00:00:00,2023-03-23 00:00:00,16,56,170,82,"""F"""
5,2023-08-12 00:00:00,2023-08-21 00:00:00,15,80,139,75,"""M"""
…,…,…,…,…,…,…,…
96,2024-11-06 00:00:00,2024-11-17 00:00:00,10,76,145,56,"""F"""
97,2024-08-09 00:00:00,2024-08-15 00:00:00,9,50,154,52,"""M"""
98,2023-12-24 00:00:00,2024-01-13 00:00:00,9,89,157,84,"""M"""


No more inconsistent rows.

This dataframe already contains a duration column, `Length_of_stay_days`, so we do not need to create a hidden helper column for this example. Instead, we convert `Length_of_stay_days` from an integer column to a Polars duration column, so metasyn recognizes it as a duration.

In [6]:
# Check the Polars documentation for the right way to convert your column to a duration.
df_duration = df.with_columns(
    pl.duration(days=pl.col("Length_of_stay_days")).alias("Length_of_stay_days")
)

df_duration.head()

Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,datetime[μs],datetime[μs],duration[μs],i64,i64,i64,str
1,2023-01-04 00:00:00,2023-01-06 00:00:00,2d,44,164,67,"""F"""
2,2023-01-08 00:00:00,2023-01-18 00:00:00,10d,78,154,57,"""F"""
3,2023-01-08 00:00:00,2023-01-19 00:00:00,11d,18,167,77,"""M"""
4,2023-01-08 00:00:00,2023-01-24 00:00:00,16d,83,177,95,"""F"""
5,2023-01-15 00:00:00,2023-01-21 00:00:00,6d,9,138,25,"""M"""


Because metasyn supports duration columns, we can define `Discharge_date` as the sum of `Admission_date` and `Length_of_stay_days`.

In [7]:
builder = MetaFrameBuilder()
builder.add_dataframe(df_duration, None)

builder["Discharge_date"].distribution = ColumnReference("Admission_date") + ColumnReference("Length_of_stay_days")

builder.fit().synthesize()

  Patient_id: 100%|██████████| 8/8 [00:00<00:00, 519.76variables/s]


Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,datetime[μs],datetime[μs],duration[μs],i64,i64,i64,str
1,2023-10-12 00:00:00,2023-10-28 00:00:00,16d,45,147,76,"""F"""
2,2024-02-15 00:00:00,2024-02-26 00:00:00,11d,44,155,81,"""F"""
3,2024-07-09 00:00:00,2024-07-12 00:00:00,3d,61,200,47,"""F"""
4,2023-09-01 00:00:00,2023-09-11 00:00:00,10d,77,155,66,"""M"""
5,2024-04-20 00:00:00,2024-04-22 00:00:00,2d,51,159,66,"""M"""
…,…,…,…,…,…,…,…
96,2023-05-24 00:00:00,2023-06-11 00:00:00,18d,61,178,31,"""M"""
97,2024-03-30 00:00:00,2024-04-05 00:00:00,6d,53,180,77,"""M"""
98,2023-07-02 00:00:00,2023-07-12 00:00:00,10d,25,162,50,"""F"""


The synthesized `Discharge_date` values now align with the corresponding `Admission_date` and `Length_of_stay_days` values.

### 2. Logical expressions and conditions

If a boolean column is fully determined by another column, you can encode that dependency directly. For example, if the dataset contains an `Adult` column, you can define it as `builder["Adult"].distribution = ColumnReference("Age") > 18`. This keeps synthesized data internally consistent without post-processing.


The comparison and logical operators let you derive boolean columns from expressions. The table below lists the implemented operators and example usage. For readability, the examples show column names directly, but in practice, reference a column with `ColumnReference()`, for example `ColumnReference("Age")`.

| Dunder | Operator | Example |
|---|---|---|
| `__add__` | `a + b` | `Age + 10` |
| `__sub__` | `a - b` | `Age - 10` |
| `__mul__` | `a * b` | `Weight_kg * 2` |
| `__truediv__` | `a / b` | `Weight_kg / 2` |
| `__pow__` | `a ** b` | `Weight_kg ** 2` |
| `__neg__` | `-a` | `-Weight_kg` |
| `__invert__` | `not a` | `not Adult` |
| `__and__` | `a & b` | `Adult & (Sex == "male")` |
| `__or__` | `a \| b` | `Adult \| Child` |
| `__eq__` | `a == b` | `Sex == "female"` |
| `__ne__` | `a != b` | `Sex != "male"` |
| `__lt__` | `a < b` | `Age < 18` |
| `__gt__` | `a > b` | `Age > 17` |
| `__le__` | `a <= b` | `Age <= 17` |
| `__ge__` | `a >= b` | `Age >= 18` |



In [8]:
# Example without relations or constraints

# Create example data with boolean "Adult" column
df = df.with_columns(
    (pl.col("Age") > 18).alias("Adult"),
)

builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

synth = builder.fit().synthesize()

n_mismatches = synth.filter(
    (pl.col("Age") > 18) & ~pl.col("Adult")
).height

print(f"\nRows where age > 18 but Adult is false: {n_mismatches} / {len(synth)}")

synth[["Age", "Adult"]]

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 581.75variables/s]


Rows where age > 18 but Adult is false: 21 / 100


Age,Adult
i64,bool
89,true
33,false
44,true
9,false
65,false
…,…
2,false
6,true
25,true


Let's fix the inconsistency.

In [9]:
# Add relations or constraints
builder["Adult"].distribution = ColumnReference("Age") > 18

synth = builder.fit().synthesize()

n_mismatches = synth.filter(
    (pl.col("Age") > 18) & ~pl.col("Adult")
).height

print(f"\nRows where age > 18 but Adult is false: {n_mismatches} / {len(synth)}")

synth[["Age", "Adult"]]

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 651.62variables/s]


Rows where age > 18 but Adult is false: 0 / 100


Age,Adult
i64,bool
79,true
80,true
14,false
48,true
72,true
…,…
37,true
39,true
52,true


#### IfThenElse example

Sometimes the distribution of a column depends on the value of another column. For example, `Height_cm` tends to differ between male and female patients. With `IfThenElse`, you can specify a different distribution or value for each group.

In [10]:
# Males: TruncatedNormal centred at 180 cm; females: centred at 170 cm
builder["Height_cm"].distribution = IfThenElse(
    ColumnReference("Sex") == "M",
    DiscreteTruncatedNormalDistribution(lower=160, upper=200, mean=180, sd=10),
    DiscreteTruncatedNormalDistribution(lower=150, upper=190, mean=170, sd=10),
)

synth = builder.fit().synthesize()
synth[["Sex", "Height_cm"]].head(10)

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 705.40variables/s]


Sex,Height_cm
str,i64
"""F""",154
"""F""",164
"""M""",179
"""M""",194
"""F""",170
"""M""",189
"""F""",178
"""F""",160
"""F""",170


Actually, this dataset includes more complex dependencies: `Height_cm` and `Weight_kg` are strongly associated with age.
For example, a 6-year-old is very unlikely to be 200 cm tall or weigh 100 kg.

To make synthetic data more realistic, we can divide age into categories and assign plausible ranges to each group:

- **0-2 years:** height 50-90 cm, weight 3-14 kg
- **3-12 years:** height 95-155 cm, weight 14-50 kg
- **13-17 years:** height 145-190 cm, weight 40-90 kg
- **18+ years:** height 145-210 cm, weight 45-150 kg

These ranges are useful defaults for synthetic data generation, but edge-case combinations can still appear.

In the example below, we combine logical and conditional operators to define distributions across `Age` ranges.

In [11]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

builder["Weight_kg"].distribution = IfThenElse(
    ColumnReference("Age") <= 2,
    DiscreteTruncatedNormalDistribution(lower=3, upper=14, mean=8, sd=2),
    IfThenElse(
        ColumnReference("Age") <= 12,
        DiscreteTruncatedNormalDistribution(lower=14, upper=50, mean=30, sd=10),
        IfThenElse(
            ColumnReference("Age") <= 17,
            DiscreteTruncatedNormalDistribution(lower=40, upper=90, mean=65, sd=10),
            DiscreteTruncatedNormalDistribution(lower=50, upper=120, mean=75, sd=15),
        ),
    ),
)

builder["Height_cm"].distribution = IfThenElse(
    ColumnReference("Age") <= 2,
    DiscreteTruncatedNormalDistribution(lower=50, upper=90, mean=70, sd=10),
    IfThenElse(
        ColumnReference("Age") <= 12,
        DiscreteTruncatedNormalDistribution(lower=95, upper=155, mean=125, sd=15),
        IfThenElse(
            ColumnReference("Age") <= 17,
            DiscreteTruncatedNormalDistribution(lower=145, upper=190, mean=167, sd=12),
            DiscreteTruncatedNormalDistribution(lower=145, upper=210, mean=172, sd=12),
        ),
    ),
)

synth = builder.fit().synthesize()

synth

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 597.82variables/s]


Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex,Adult
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,str,bool
1,2024-02-19 00:00:00,2023-02-08 00:00:00,13,38,169,69,"""F""",true
2,2024-02-08 00:00:00,2024-03-22 00:00:00,12,39,167,64,"""F""",true
3,2023-11-17 00:00:00,2024-11-12 00:00:00,12,52,167,71,"""M""",true
4,2023-02-06 00:00:00,2024-08-21 00:00:00,4,79,171,80,"""M""",false
5,2023-11-12 00:00:00,2023-05-07 00:00:00,12,31,166,51,"""F""",false
…,…,…,…,…,…,…,…,…
96,2023-02-11 00:00:00,2024-02-22 00:00:00,18,5,128,25,"""M""",true
97,2023-03-30 00:00:00,2024-08-23 00:00:00,1,38,159,77,"""F""",true
98,2024-06-20 00:00:00,2023-07-11 00:00:00,13,73,177,60,"""M""",true


As you can see, height and weight are adjusted to match the different age groups in the synthesized data.

### 3. Important note

When defining relationships between columns, it is usually safer to **preserve the broad structure of the data** than to reproduce every numerical detail exactly. Metasyn can help reduce disclosure risk through its modelling choices and constraints, but it cannot prevent users from encoding **sensitive information indirectly through overly specific rules**. Aim for relationships that are realistic enough for the intended use case, while still remaining approximate.